# 🚀 DataMate — Train Qwen2.5-Coder-7B LoRA (890 Mẫu Verified)

Notebook huấn luyện mô hình **Qwen2.5-Coder-7B-Instruct** với bộ dataset SFT chuẩn hóa **890 mẫu** (`verified_sft_dataset.jsonl`) trên Google Colab A100/L4/T4.
- **Tập dữ liệu**: 890 mẫu AST-valid, phủ 24 doanh nghiệp, kiểm tra chéo 100% khớp với gold answers.
- **Cơ chế Loss**: Completion-only loss (chỉ tính gradient trên mã code Python của assistant, mask prompt).
- **Adapter Output**: Lưu trực tiếp vào Google Drive (`result/sft_adapter_qwen7b_890/final_adapter`).
- **Đánh giá PA**: Tự động đo điểm Penalized Accuracy trên 30 câu hỏi Pilot ngay sau khi train.

## 1. Kết nối Google Drive & Thiết lập môi trường

In [ ]:
from google.colab import drive
import os, sys

drive.mount('/content/drive')
PROJECT_DIR = '/content/drive/MyDrive/Colab Notebooks/ZALO-AI-CHALLENGE-2026'
os.chdir(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)
print(f"📂 Thư mục dự án: {os.getcwd()}")

## 2. Cài đặt thư viện & Kiểm tra GPU

In [ ]:
!pip install -q transformers peft accelerate datasets bitsandbytes calamine
!nvidia-smi

## 3. Kiểm tra Base Model Qwen2.5-Coder-7B-Instruct

In [ ]:
import os
from huggingface_hub import snapshot_download

BASE_MODEL_DIR = "/content/models/Qwen2.5-Coder-7B-Instruct"
if not os.path.exists(os.path.join(BASE_MODEL_DIR, "config.json")):
    print(f"⏳ Base model chưa có tại {BASE_MODEL_DIR}. Bắt đầu tải từ Hugging Face...")
    snapshot_download(
        repo_id="Qwen/Qwen2.5-Coder-7B-Instruct",
        local_dir=BASE_MODEL_DIR,
        ignore_patterns=["*.pth", "*.pt", "*.bin"]
    )
    print(f"✅ Tải xong Qwen2.5-Coder-7B-Instruct vào {BASE_MODEL_DIR}")
else:
    print(f"✅ Base model đã có sẵn tại {BASE_MODEL_DIR}!")

## 4. Huấn luyện SFT LoRA (890 mẫu Verified)

In [ ]:
# Huấn luyện SFT LoRA với 890 mẫu verified
# Trên A100: batch_size=2, grad_accum=4 (effective batch=8), lr=1e-4, 3 epochs (~15-20 phút)
!python scripts/train_sft_lora.py \
    --model "/content/models/Qwen2.5-Coder-7B-Instruct" \
    --data "verified_sft_dataset.jsonl" \
    --output_dir "result/sft_adapter_qwen7b_890" \
    --epochs 3.0 \
    --batch_size 2 \
    --grad_accum 4 \
    --lr 1e-4 \
    --max_seq_len 8192

## 5. Đánh giá Penalized Accuracy (PA) trên 30 câu Pilot

In [ ]:
# Đo điểm PA trực tiếp của Adapter 890 mẫu vừa train trên 30 câu hỏi Pilot
!python scripts/evaluate_pilot_30.py \
    --adapter-path "result/sft_adapter_qwen7b_890/final_adapter" \
    --output-dir "result/eval_qwen7b_890_pilot"